Client ---> POST /auth/signup {email, password}

        ||

Auth Service: 1. Email Check kro (already exists?) 2. Password hash 3. User record banao(id, email, passwordHash) 4. JWT token generate 5. Event emit kro: "user.registered" {userId, email} 6. Response bhejo: {token, user: {id, email }}

        ||

Client <-- 201 Created {token, user}
||
(Background mein) : Event Bus -> user.registered
User Service consume event: 1. Naya UserProfile row banaye (userId, email) 2. Default values set kre 3. Ready for profile updated


Client --> POST /auth/login {email, password}
||
Auth Service: 1. Email se user dhundo 2. Password verify kro (bcrype, argon) 3. JWT token generate 4. Response bhejo: {token: {id, email}}
|
Client <--- 200 ok {token, user}


Client → PUT /users/me { firstName, bio }
Headers: Authorization: Bearer <JWT>
↓
User Service:

1. JWT verify kare (locally, public key se)
2. userId nikale token se
3. Profile update kare
4. Response bheje: { id, firstName, bio, ... }
   ↓
   Client ← 200 OK { updated profile }


Client → GET /users/:id
↓
User Service:

1. JWT verify kare (agar protected hai)
2. Database se profile fetch kare
3. Response bheje: { id, firstName, bio, imageUrl, ... }
   ↓
   Client ← 200 OK { profile }


Events:
Event || Emit || By Consume By || Kaam
user.registered | Auth | User | Profile row banaye
user.deleted | Auth | User | Profile delete kare
user.email.verified | Auth |User |Profile mein flag update
user.password.changed | Auth | User |(optional) notification
user.profile.updated | User | Auth | (optional) email sync


event tools:
rabbitmq
kafka
redis pub/sub
nats
bull mq

Client
  │
  │  POST /api/users/me/profile
  │  Headers: Authorization: Bearer <JWT>
  │  Body: { firstName: "Daniyal", bio: "Dev" }
  ▼
┌─────────────────────────────────────────┐
│         API GATEWAY                     │
│                                         │
│  1. ✅ JWT signature verify             │
│  2. ✅ Token expire check               │
│  3. ✅ Rate limit check                 │
│  4. ✅ Route match (/api/users → User)  │
│  5. ✅ Request ID generate              │
│  6. ✅ Log request                      │
│  7. ✅ Add headers:                     │
│       X-User-Id: <from JWT>             │
│       X-User-Role: <from JWT>           │
│       X-Request-Id: <uuid>              │
│                                         │
│  ❌ Body validate NAHI karega            │
│  ❌ Business logic NAHI karega           │
└────────────────┬────────────────────────┘
                 │
                 │  Forward with headers
                 ▼
┌─────────────────────────────────────────┐
│         USER SERVICE                    │
│                                         │
│  1. ✅ X-User-Id header trust kare       │
│     (kyunki gateway ne verify kiya)     │
│  2. ✅ Body validate kare (Zod)          │
│  3. ✅ Business logic chalao             │
│  4. ✅ DB update kare                    │
│  5. ✅ Response bheje                    │
└────────────────┬────────────────────────┘
                 │
                 ▼
              Client

                        ┌─────────────┐
                        │   Client    │
                        └──────┬──────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │    API GATEWAY       │
                    │                      │
                    │  • JWT verify        │
                    │  • Rate limit        │
                    │  • Routing           │
                    │  • Logging           │
                    │  • CORS              │
                    └──────┬───────────────┘
                           │
          ┌────────────────┼────────────────┐
          │                │                │
          ▼                ▼                ▼
    ┌──────────┐    ┌──────────┐    ┌──────────┐
    │  AUTH    │    │  USER    │    │  OTHER   │
    │ SERVICE  │    │ SERVICE  │    │ SERVICE  │
    │          │    │          │    │          │
    │ • login  │    │ • profile│    │          │
    │ • signup │    │ • search │    │          │
    │ • refresh│    │ • prefs  │    │          │
    │ • JWT gen│    │ • JWT    │    │ • JWT    │
    │          │    │   consume│    │   consume│
    └────┬─────┘    └────┬─────┘    └────┬─────┘
         │               │               │
         │  Events       │               │
         └───────┬───────┘               │
                 ▼                       │
         ┌───────────────┐               │
         │  EVENT BUS    │               │
         └───────────────┘               │
                 │                       │
         ┌───────┴───────┐               │
         ▼               ▼               ▼
    ┌─────────┐    ┌─────────┐    ┌─────────┐
    │ Auth DB │    │ User DB │    │ Other DB│
    └─────────┘    └─────────┘    └─────────┘

┌─────────────────────────────────────────────────┐
│ 1. ROUTE                                        │
│    router.post("/signup", validate(signupSchema),│
│                authController.signup);          │
│                                                 │
│    ↑ Yahan VALIDATION middleware chalta hai     │
│      (runtime pe check)                         │
└────────────────────┬────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────┐
│ 2. MIDDLEWARE (validation)                      │
│                                                 │
│    validate(schema) {                           │
│      const data = schema.parse(req.body);      │
│      // ✅ Runtime check — data sahi hai?        │
│      req.body = data; // cleaned data           │
│      next();                                    │
│    }                                            │
└────────────────────┬────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────┐
│ 3. CONTROLLER                                   │
│                                                 │
│    export const signup = async (req, res) => { │
│      // Types ab available hain                 │
│      const data: SignupInput = req.body;        │
│      //       ↑ TYPE (compile time)             │
│                                                 │
│      const result = await authService.signup(data);│
│      res.json(result);                          │
│    };                                           │
└────────────────────┬────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────┐
│ 4. SERVICE                                      │
│                                                 │
│    export const signup = async (data: SignupInput): │
│                                    Promise<AuthResponse> => {│
│      // ↑ TYPES use ho rahi hain                │
│                                                 │
│      const existing = await authRepo.findByEmail(data.email);│
│      // ↑ REPOSITORY ka type                    │
│                                                 │
│      if (existing) throw new ApiError(409, "Email exists");│
│                                                 │
│      const passwordHash = await bcrypt.hash(data.password, 12);│
│      const user = await authRepo.create({ ...data, passwordHash });│
│                                                 │
│      return {                                   │
│        user: { id: user.id, email: user.email, role: user.role },│
│        ...await generateTokens(user),           │
│      };                                         │
│    };                                           │
└────────────────────┬────────────────────────────┘
                     │
                     ▼
┌─────────────────────────────────────────────────┐
│ 5. REPOSITORY                                   │
│                                                 │
│    export const create = async (data: CreateUserInput): │
│                                      Promise<User> => {│
│      return db.orm.public.User.create({ data });│
│    };                                           │
└─────────────────────────────────────────────────┘